# Lily Vision Phase 1 Projector Alignment — Modal A100 40GB
**Trains 2-layer MLP projector connecting SigLIP-2 Vision Encoder to Lily 1.5B LLM**

Freezes SigLIP-2 vision tower and Lily 1.5B LLM base weights. Trains *only* the 2-layer linear projection bottleneck (with 18x18 adaptive average pooling downsampling 729 vision tokens to 324 tokens) on 120k image-caption alignment pairs.

## Cell 1 — Install Dependencies

In [ ]:
# ==============================================================================
# Cell 1 — Dependency Installation
# ==============================================================================
%uv pip install torch transformers accelerate datasets wandb pillow -q

## Cell 2 — Hardware Probe & Hardware Check

In [ ]:
# ==============================================================================
# Cell 2 — GPU Compute & VRAM Verification
# ==============================================================================
import torch
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM: {p.total_memory/1e9:.1f} GB | Compute: cc={p.major}.{p.minor}")
assert p.major >= 8, "Requires Ampere GPU (A100)"

## Cell 3 — Configuration & Alignment Hyperparameters

In [ ]:
# ==============================================================================
# Cell 3 — Vision-Language Configuration & Hyperparameters
# ==============================================================================
import os
HF_USERNAME = "abhinav0231"

VISION_ENCODER = "google/siglip-so400m-patch14-384"  # 1152-dim vision features
LLM_MODEL      = f"{HF_USERNAME}/Lily-1.5b-v0.3"      # 1536-dim LLM hidden state
ALIGN_DATASET  = f"{HF_USERNAME}/lily-vision-align-120k"
PROJECTOR_REPO = f"{HF_USERNAME}/lily-vision-projector-v5"

# Phase 1 Hyperparameters
LEARNING_RATE = 2e-4  # Higher learning rate for initial random MLP initialization
BATCH_SIZE    = 16    # Micro-batch per device
GRAD_ACCUM    = 3     # Effective Batch Size = 16 * 3 = 48
NUM_EPOCHS    = 1

print(f"Vision Encoder : {VISION_ENCODER}")
print(f"LLM Backbone   : {LLM_MODEL}")
print(f"Effective Batch: {BATCH_SIZE * GRAD_ACCUM}")

## Cell 4 — Define `LilyVisionProjector` Architecture

In [ ]:
# ==============================================================================
# Cell 4 — 2-Layer MLP Bottleneck Projector with 18x18 Adaptive Pooling
# Downsamples 729 vision tokens (27x27 grid) to 324 tokens (18x18 grid)
# and maps 1152 vision features to 1536 LLM hidden dimensions.
# ==============================================================================
import torch.nn as nn

class LilyVisionProjector(nn.Module):
    def __init__(self, vision_dim=1152, llm_dim=1536):
        super().__init__()
        self.downsampler = nn.AdaptiveAvgPool2d((18, 18))  # 18x18 = 324 tokens
        self.mlp = nn.Sequential(
            nn.Linear(vision_dim, 2048),
            nn.SiLU(),
            nn.Linear(2048, llm_dim)
        )

    def forward(self, x):
        # x shape: [B, 729, 1152]
        B, N, C = x.shape
        H = W = int(N ** 0.5)  # 27x27 grid
        x_2d = x.transpose(1, 2).view(B, C, H, W)
        x_pooled = self.downsampler(x_2d).flatten(2).transpose(1, 2)  # [B, 324, 1152]
        return self.mlp(x_pooled)  # [B, 324, 1536]

projector = LilyVisionProjector().cuda()
print("✅ LilyVisionProjector initialized")

## Cell 5 — Layer Freezing Logic

In [ ]:
# ==============================================================================
# Cell 5 — Freeze Vision Encoder & LLM Base Weights
# In Phase 1, ONLY the 2-layer MLP projector requires gradients.
# ==============================================================================
print("✅ Layer freezing applied: Vision Encoder = Frozen | LLM = Frozen | Projector = Trainable")